# Task 2: Adversarial Search - Tank Battle Game
**Môn học:** Introduction to Artificial Intelligence (503043) - Trường Đại học Tôn Đức Thắng (TDTU)  
**Thành viên:** Nguyễn Hữu Tân (524H0028) & Nguyễn Đức Hiếu (524H0087)

## Mục lục
1. **Mô hình hóa Zero-Sum Game & Kiểm tra Vật lý Đạn (Cell-by-Cell Collision)**
2. **So sánh 4 Cấu hình Thuật toán (Minimax, Alpha-Beta, Heuristic Minimax, Heuristic Alpha-Beta)**
3. **Cơ chế Nạp Thuật toán Ngoài (Dynamic External Agent Plugin Loader)**

### 1. Khởi tạo Trạng thái Game và Kiểm tra Luật Vật lý (HP & Cell-by-Cell Bullet Physics)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from task2_tank.engine.entities import Tank, Base, Wall, Bullet
from task2_tank.engine.actions import Action, Direction
from task2_tank.engine.maps import MapBuilder
from task2_tank.engine.rules import GameRules

state = MapBuilder.create_standard_map()
print(f"[*] Kích thước bản đồ: {state.width}x{state.height}")
print(f"[*] Máu ban đầu (Initial HP): Tank = {Tank.INITIAL_HP}, Base = {Base.INITIAL_HP}, Wall = {Wall.INITIAL_HP}")
print(f"[*] Tốc độ đạn (Bullet Speed): {Bullet.SPEED} ô / chu kỳ (kiểm tra va chạm từng ô một)")
print(f"[*] Danh sách 06 hành động hợp lệ: {[a.name for a in state.get_legal_actions(0)]}")

next_state = GameRules.apply_action(state, Action.FIRE)
print(f"[*] Sau khi Player 0 bắn đạn: Số đạn đang bay = {len(next_state.bullets)}, Lượt tiếp theo = Player {next_state.current_player}")


[*] Kích thước bản đồ: 13x13
[*] Máu ban đầu (Initial HP): Tank = 6, Base = 9, Wall = 3
[*] Tốc độ đạn (Bullet Speed): 3 ô / chu kỳ (kiểm tra va chạm từng ô một)
[*] Danh sách 06 hành động hợp lệ: ['MOVE_UP', 'MOVE_DOWN', 'MOVE_LEFT', 'MOVE_RIGHT', 'FIRE', 'STAY']
[*] Sau khi Player 0 bắn đạn: Số đạn đang bay = 1, Lượt tiếp theo = Player 1


### 2. Thực nghiệm So sánh 4 Cấu hình Thuật toán Tìm kiếm Đối kháng

In [2]:
from task2_tank.experiments.benchmark_tanks import TankTournament

tournament = TankTournament(max_cycles_per_game=30)
tournament.run_single_move_benchmark(depths=[1, 2, 3])


=== 1. RUNNING SINGLE-MOVE SEARCH BENCHMARK ===
-> [Depth 1] 1. Minimax Search                | Time:  0.00026s | Nodes:     12 | Prunes:    0 | Move: MOVE_UP
-> [Depth 1] 2. Alpha-Beta Search             | Time:  0.00015s | Nodes:     12 | Prunes:    0 | Move: MOVE_UP
-> [Depth 1] 3. Heuristic Minimax             | Time:  0.00026s | Nodes:     12 | Prunes:    0 | Move: MOVE_UP
-> [Depth 1] 4. Heuristic Alpha-Beta          | Time:  0.00023s | Nodes:     12 | Prunes:    0 | Move: MOVE_UP
-> [Depth 2] 1. Minimax Search                | Time:  0.00060s | Nodes:     48 | Prunes:    0 | Move: MOVE_UP
-> [Depth 2] 2. Alpha-Beta Search             | Time:  0.00033s | Nodes:     23 | Prunes:    5 | Move: MOVE_UP
-> [Depth 2] 3. Heuristic Minimax             | Time:  0.00114s | Nodes:     48 | Prunes:    0 | Move: FIRE
-> [Depth 2] 4. Heuristic Alpha-Beta          | Time:  0.00081s | Nodes:     28 | Prunes:    4 | Move: FIRE
-> [Depth 3] 1. Minimax Search                | Time:  0.00368s | Node

[{'depth': 1,
  'agent': '1. Minimax Search',
  'avg_time': 0.00025643333113597083,
  'avg_nodes': 12.0,
  'avg_prunes': 0.0,
  'action': 'MOVE_UP'},
 {'depth': 1,
  'agent': '2. Alpha-Beta Search',
  'avg_time': 0.0001464000015403144,
  'avg_nodes': 12.0,
  'avg_prunes': 0.0,
  'action': 'MOVE_UP'},
 {'depth': 1,
  'agent': '3. Heuristic Minimax',
  'avg_time': 0.00026019999738006544,
  'avg_nodes': 12.0,
  'avg_prunes': 0.0,
  'action': 'MOVE_UP'},
 {'depth': 1,
  'agent': '4. Heuristic Alpha-Beta',
  'avg_time': 0.00023176666582003236,
  'avg_nodes': 12.0,
  'avg_prunes': 0.0,
  'action': 'MOVE_UP'},
 {'depth': 2,
  'agent': '1. Minimax Search',
  'avg_time': 0.0006012333324179053,
  'avg_nodes': 48.0,
  'avg_prunes': 0.0,
  'action': 'MOVE_UP'},
 {'depth': 2,
  'agent': '2. Alpha-Beta Search',
  'avg_time': 0.00032946666760835797,
  'avg_nodes': 23.0,
  'avg_prunes': 5.0,
  'action': 'MOVE_UP'},
 {'depth': 2,
  'agent': '3. Heuristic Minimax',
  'avg_time': 0.0011403000001640369,
 

### 3. Kiểm thử Cơ chế Tích hợp Thuật toán Nhóm Khác (External Agent Loader)

In [3]:
from task2_tank.agent_loader import AgentLoader

ext_dir = os.path.join(os.getcwd(), "agents", "external")
discovered = AgentLoader.discover_external_agents(ext_dir)
print(f"[*] Số lượng External Agent tìm thấy: {len(discovered)}")
for ext_agent in discovered:
    act = ext_agent.choose_action(state.clone(), player_id=1)
    print(f"  - Agent: {ext_agent.name} -> Quyết định hành động: {act.name}")


[AgentLoader] Successfully loaded external agent 'RivalGroup-HeuristicBot' from C:\Users\PC\Desktop\Ai_final\Final_AI_BanSung\source\task2_tank\agents\external\sample_rival_agent.py
[*] Số lượng External Agent tìm thấy: 1
  - Agent: RivalGroup-HeuristicBot -> Quyết định hành động: FIRE
